In [ ]:
"""
Flow Logs to Bronze - OPTIMIZED (No Full Scan Each Run)
Uses a "source file catalog" to avoid rescanning all files
"""

from pyspark.sql.functions import current_timestamp, input_file_name, sha2, col, lit
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, BooleanType
from datetime import datetime

spark.conf.set("spark.sql.files.ignoreMissingFiles", "true")
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
spark.conf.set("spark.sql.adaptive.enabled", "true")

# ============================================================================
# CONFIGURATION
# ============================================================================

FLOW_LOGS_SOURCE = "file:/Volumes/security_lake/default/flow_logs"
BRONZE_DATA_PATH = "/Volumes/gitrepo/default/git_oci_aidp_bronze/flow_logs/data"
CHECKPOINT_PATH = "/Volumes/gitrepo/default/git_oci_aidp_bronze/flow_logs/manual_checkpoint"

# NEW: Source catalog to avoid full scans
SOURCE_CATALOG_PATH = "/Volumes/gitrepo/default/git_oci_aidp_bronze/flow_logs/source_catalog"

BATCH_SIZE = 5000

print("=" * 70)
print("FLOW LOGS TO BRONZE - OPTIMIZED")
print("=" * 70)
print(f"Batch Size: {BATCH_SIZE} files")
print("=" * 70)

# ============================================================================
# STEP 1: BUILD OR UPDATE SOURCE CATALOG (ONE TIME)
# ============================================================================

print("\n[STEP 1] Checking source catalog...")

catalog_exists = False
try:
    catalog_df = spark.read.parquet(SOURCE_CATALOG_PATH)
    total_in_catalog = catalog_df.count()
    catalog_exists = True
    print(f"✓ Source catalog exists: {total_in_catalog} files")
except:
    print("✓ No source catalog found - will create it")

# Check if we need to update catalog (only if it doesn't exist or you want to refresh)
if not catalog_exists:
    print("\n[Building source catalog - ONE TIME ONLY]")
    print("This will scan all files once and save the list...")
    
    import time
    start = time.time()
    
    all_files_df = (spark.read.text(f"{FLOW_LOGS_SOURCE}/**/*.log.gz")
                    .select(input_file_name().alias("file_path"))
                    .distinct()
                    .withColumn("discovered_at", current_timestamp())
                    .withColumn("processed", lit(False)))
    
    file_count = all_files_df.count()
    
    # Save catalog
    all_files_df.write.mode("overwrite").parquet(SOURCE_CATALOG_PATH)
    
    scan_time = time.time() - start
    print(f"✓ Catalog built: {file_count} files in {scan_time:.1f}s")
    print("✓ This was a ONE-TIME scan - future runs will be FAST")
    
    catalog_df = all_files_df

# ============================================================================
# STEP 2: GET PROCESSED FILES FROM CHECKPOINT
# ============================================================================

print("\n[STEP 2] Loading processed files...")

processed_files = set()

try:
    checkpoint_df = spark.read.parquet(CHECKPOINT_PATH)
    processed_files = set([row.file_path for row in checkpoint_df.collect()])
    print(f"✓ Checkpoint: {len(processed_files)} files processed")
except:
    print("✓ No checkpoint yet")

# ============================================================================
# STEP 3: FIND UNPROCESSED FILES (FAST - No scan!)
# ============================================================================

print("\n[STEP 3] Finding unprocessed files (no scan!)...")

# Filter catalog for unprocessed files
unprocessed_df = catalog_df.filter(~col("file_path").isin(list(processed_files)))

new_files = [row.file_path for row in unprocessed_df.limit(BATCH_SIZE).collect()]

total_files = catalog_df.count()
remaining = total_files - len(processed_files)

print(f"✓ Total files: {total_files}")
print(f"✓ Already processed: {len(processed_files)}")
print(f"✓ Remaining: {remaining}")
print(f"✓ Will process: {len(new_files)} files this run")

# ============================================================================
# STEP 4: PROCESS FILES
# ============================================================================

if not new_files:
    print("\n" + "=" * 70)
    print("✓ ALL FILES PROCESSED!")
    print("=" * 70)
    
    try:
        bronze_df = spark.read.parquet(BRONZE_DATA_PATH)
        print(f"Total bronze: {bronze_df.count():,} records")
    except:
        pass
    
    print("\n📋 Move to Silver layer processing")
    
else:
    print(f"\n[STEP 4] Reading {len(new_files)} files...")
    
    import time
    start = time.time()
    
    df = (spark.read.text(new_files)
          .withColumnRenamed("value", "raw_json")
          .withColumn("source_file", input_file_name())
          .withColumn("ingest_time", current_timestamp())
          .withColumn("event_hash", sha2(col("raw_json"), 256)))
    
    record_count = df.count()
    read_time = time.time() - start
    print(f"✓ Read {record_count:,} records in {read_time:.1f}s")
    
    print("\n[STEP 5] Writing to bronze...")
    start = time.time()
    df.write.mode("append").parquet(BRONZE_DATA_PATH)
    write_time = time.time() - start
    print(f"✓ Written in {write_time:.1f}s")
    
    print("\n[STEP 6] Updating checkpoint...")
    schema = StructType([
        StructField("file_path", StringType(), False),
        StructField("processed_timestamp", TimestampType(), False)
    ])
    
    data = [(fp, datetime.now()) for fp in new_files]
    checkpoint_df = spark.createDataFrame(data, schema)
    checkpoint_df.write.mode("append").parquet(CHECKPOINT_PATH)
    print("✓ Checkpoint updated")
    
    # Stats
    bronze_df = spark.read.parquet(BRONZE_DATA_PATH)
    total_records = bronze_df.count()
    
    print("\n" + "=" * 70)
    print("✓ COMPLETED")
    print("=" * 70)
    print(f"  This run: {len(new_files)} files, {record_count:,} records, {read_time + write_time:.1f}s")
    print(f"  Bronze total: {total_records:,} records")
    print(f"  Remaining: {remaining - len(new_files)} files")
    
    if remaining > len(new_files):
        remaining_runs = (remaining - len(new_files)) // BATCH_SIZE + 1
        est_time = remaining_runs * (read_time + write_time) / 60
        print(f"  Estimated: {remaining_runs} more runs (~{est_time:.0f} minutes)")
    
    print("=" * 70)
    
    if remaining > len(new_files):
        print(f"\n📋 Run again to process next batch")
    else:
        print("\n📋 Next run will complete!")

print("\n✓ Done")